<h3>Voting Classifier</h3>

<p>
A Voting Classifier is an ensemble learning method that combines the predictions
of multiple classifiers and makes a final prediction using voting.
</p>

<h4>Procedure</h4>
<ol>
    <li>Select multiple classifiers (e.g., LR, DT, SVM).</li>
    <li>Train all classifiers on the training data.</li>
    <li>Each classifier makes a prediction.</li>
    <li>Combine predictions using hard voting or soft voting.</li>
    <li>Return the final predicted class.</li>
</ol>

In [15]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score
import pandas as pd
import numpy as np

In [16]:
iris=load_iris()

In [17]:
df=pd.DataFrame(iris.data,columns=iris.feature_names)
df['target']=iris.target

In [18]:
X=df.iloc[:,0:2]
y=df.iloc[:,-1]

In [19]:
# base classifiers
lr=LogisticRegression()
dt=DecisionTreeClassifier()
svc=SVC(probability=True)

In [20]:
# list of (str,estimators) tuples
estimators=[('lr',lr),('dt',dt),('svc',svc)]



Instead of splitting the dataset once into train and test sets, it splits the data K times and tests the model on different portions of the data.

## Cross-validation 
A resampling technique used to evaluate a machine learning model by dividing the dataset into K folds, training the model on K−1 folds, testing it on the remaining fold, and averaging the performance across all folds.
<p>Syntax : cross_val_score(model, X, y, cv=5, scoring='f1_macro')</p>

In [21]:
for estimator in estimators:
    # return an array of accuracies at different folds
    x=cross_val_score(estimator[1],X,y,cv=10,scoring='accuracy')
    print(estimator[0],np.round(np.mean(x)*100,2))



lr 80.67
dt 67.33
svc 82.67


<h3>What Happens During Cross Validation?</h3>

<p><strong>Suppose <code>cv = 10</code>.</strong></p>

<ul>
    <li>The dataset is split into <strong>10 folds</strong>.</li>
    <li>For <strong>Fold 1</strong>:
        <ul>
            <li><code>VotingClassifier</code> is trained on the other <strong>9 folds</strong>.</li>
            <li>Internally, all base models (e.g., Logistic Regression, Decision Tree, SVM) are fitted on those 9 folds.</li>
            <li><code>VotingClassifier</code> makes predictions on the remaining fold.</li>
            <li>Accuracy is recorded.</li>
        </ul>
    </li>
    <li>The same process is repeated for all <strong>10 folds</strong>.</li>
    <li>The final cross-validation score is the <strong>average accuracy across all 10 folds</strong>.</li>
</ul>

In [22]:
# hard voting
vc=VotingClassifier(estimators=estimators,voting='hard')
x=cross_val_score(vc,X,y,cv=10,scoring='accuracy')
print(np.round(np.mean(x),2))

0.81


In [23]:
# soft voting
vc=VotingClassifier(estimators=estimators,voting='soft')
x=cross_val_score(vc,X,y,cv=10,scoring='accuracy')
print(np.round(np.mean(x),2))

0.71


<h3>Difference Between Hard Voting and Soft Voting</h3>

<table border="1">
    <tr>
        <th>Hard Voting</th>
        <th>Soft Voting</th>
    </tr>
    <tr>
        <td>Uses predicted class labels.</td>
        <td>Uses predicted probabilities.</td>
    </tr>
    <tr>
        <td>Final class is chosen by majority vote.</td>
        <td>Final class is chosen by highest average probability.</td>
    </tr>
    <tr>
        <td>Simpler and faster.</td>
        <td>Usually gives better performance.</td>
    </tr>
    <tr>
        <td>Does not consider confidence of predictions.</td>
        <td>Considers confidence of predictions.</td>
    </tr>
</table>

In [32]:
from sklearn.svm import SVC
sv1=SVC(probability=True,kernel='poly',degree=1)
sv2=SVC(probability=True,kernel='poly',degree=2)
sv3=SVC(probability=True,kernel='poly',degree=3)
sv4=SVC(probability=True,kernel='poly',degree=4)
sv5=SVC(probability=True,kernel='poly',degree=5)

estimators=[('sv1',sv1),('sv2',sv2),('sv3',sv3),('sv4',sv4),('sv5',sv5)]

# testing each base model sparately
for estimator in estimators:
    x=cross_val_score(estimator[1],X,y,cv=10,scoring='accuracy')
    print(estimator[0],np.round(np.mean(x),4))

sv1 0.82
sv2 0.8133
sv3 0.8067
sv4 0.8
sv5 0.7867


In [ ]:
# combining these base models to get better result
vc=VotingClassifier(estimators=estimators,voting='hard')
x=cross_val_score(vc,X,y,cv=10,scoring='accuracy')
print(np.round(np.mean(x),4)*100,"%")

81.33 %
